# Setup

In [1]:
import torch
print(torch.__version__)
print(torch.__file__)

/opt/anaconda3/envs/ml/lib/python3.10/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


2.2.2
/opt/anaconda3/envs/ml/lib/python3.10/site-packages/torch/__init__.py


In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

# Use GPU if available
device = 0 if torch.cuda.is_available() else -1

# Load multilingual sentiment model
sentiment_model = pipeline(
    "text-classification",
    model="tabularisai/multilingual-sentiment-analysis",
    tokenizer="tabularisai/multilingual-sentiment-analysis",
    device=device,
    truncation=True,
    max_length=512
)

In [ ]:
# Batch sentiment function
def multilingual_sentiment(texts, batch_size=32):
    # Replace non-string values with empty strings
    texts = ["" if not isinstance(t, str) else t for t in texts]

    labels = []
    scores = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]

        results = sentiment_model(
            batch,
            truncation=True,
            max_length=512
        )

        labels.extend([r["label"] for r in results])
        scores.extend([r["score"] for r in results])

    return labels, scores

In [ ]:
# read the data in 
teaser = pd.read_csv("./data/teaser_comments.csv")
debut = pd.read_csv("./data/debut_comments.csv")

# Run Model On TEASER Data

In [ ]:
BATCH = 64

labels, scores = multilingual_sentiment(teaser["comment_text"].tolist(), batch_size=BATCH)

teaser["multilingual_sentiment"] = labels
teaser["multilingual_confidence"] = scores

print(teaser["multilingual_sentiment"].value_counts())
print(teaser["multilingual_confidence"].median())

# Run Model On DEBUT Data

In [ ]:
labels, scores = multilingual_sentiment(debut["comment_text"].tolist(), batch_size=BATCH)

debut["multilingual_sentiment"] = labels
debut["multilingual_confidence"] = scores

print(debut["multilingual_sentiment"].value_counts())
print(debut["multilingual_confidence"].median())